# Semana 1: carga y reporte de Fast_Food_Restaurants

En este notebook se construye un flujo completo de trabajo:

1. Configurar la ruta del proyecto para importar las clases reutilizables.
2. Cargar `Fast_Food_Restaurants` desde Hugging Face.
3. Obtener un `DataFrame` de pandas y revisar su estructura.
4. Generar un reporte en pantalla y guardarlo en un archivo `.txt`.

Ejecuta las celdas en orden. Si el dataset requiere autenticación, inicia sesión previamente con `hf auth login`.

## 1. Preparar el entorno e importar la clase de carga

El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar `src/clases_po` y se agrega `src` a `sys.path`. Así se puede importar `HFDatasetLoader` sin instalar el proyecto como paquete.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases import HFDatasetLoader

/home/runner/work/analisis_datos_dataframe_Hugging_Face/analisis_datos_dataframe_Hugging_Face/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configurar el dataset y la salida

`dataset_name` identifica el repositorio en Hugging Face y `split="train"` indica la división que se descargará. El parámetro `filename` hace que el `DataFrame` también se guarde como CSV en `src/sincronicas/datos/` cuando el notebook se ejecuta desde esa carpeta.

In [2]:
loader = HFDatasetLoader(
    dataset_name="cs-uche/car_dealership",
    split="train",
    filename="datos/car_dealership.csv",
)


## 3. Descargar el dataset y crear el DataFrame

El método `load()` descarga el dataset, lo convierte a `pandas.DataFrame` y lo asigna a la variable `df`. La expresión `df.head()` muestra las primeras filas para comprobar que la carga fue exitosa.

In [3]:
df = loader.load()
df.head()

Generating train split:   0%|          | 0/15389 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 15389/15389 [00:00<00:00, 468441.92 examples/s]

Generating validation split:   0%|          | 0/15389 [00:00<?, ? examples/s]

Generating validation split: 100%|██████████| 15389/15389 [00:00<00:00, 499146.60 examples/s]

,Prod. year,Levy,Mileage,Engine volume,Manufacturer,Model,Fuel type,Leather interior,Gear box type,Category,Price
0,2013,779,225510.0,2.5,TOYOTA,Camry,Hybrid,Yes,Automatic,Sedan,314
1,2007,1282,76198.0,2.0,CHEVROLET,Captiva,Diesel,Yes,Automatic,Jeep,6429
2,2010,1399,189530.0,3.5,MERCEDES-BENZ,E 350,Diesel,Yes,Automatic,Sedan,12388
3,2012,642,218525.0,2.0,CHEVROLET,Orlando,Diesel,Yes,Automatic,Jeep,14834
4,2000,2146,25000.0,3.0,BMW,X5,LPG,Yes,Tiptronic,Jeep,10036


## 4. Revisar la estructura del DataFrame

Esta celda realiza una revisión inicial: cuenta filas y columnas y muestra el tipo de cada columna junto con información de memoria y valores no nulos. Esta inspección ayuda a detectar problemas antes de analizar los datos.

In [4]:
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
df.info()

Filas: 15389
Columnas: 11
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15389 entries, 0 to 15388
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Prod. year        15389 non-null  int64  
 1   Levy              15389 non-null  int64  
 2   Mileage           15389 non-null  float64
 3   Engine volume     15389 non-null  float64
 4   Manufacturer      15389 non-null  object 
 5   Model             15389 non-null  object 
 6   Fuel type         15389 non-null  object 
 7   Leather interior  15389 non-null  object 
 8   Gear box type     15389 non-null  object 
 9   Category          15389 non-null  object 
 10  Price             15389 non-null  int64  
dtypes: float64(2), int64(3), object(6)
memory usage: 1.3+ MB


## 5. Importar la clase de reportes

`Reportes` recibe un `DataFrame` en sus métodos y permite generar un resumen textual, imprimirlo en la consola o guardarlo en un archivo `.txt`.

In [5]:
from clases import Reportes

## 6. Generar y visualizar el reporte como texto

`generar_reporte(df)` devuelve un `str` con el número de filas, columnas, duplicados, tipos de datos, valores nulos y estadísticas numéricas. En esta celda el texto se guarda temporalmente en `texto` y se imprime.

In [6]:
reportes = Reportes()
texto = reportes.generar_reporte(df)

print(texto)

REPORTE DEL DATAFRAME
Filas: 15389
Columnas: 11
Duplicados: 2579

Columnas y tipos:
- Prod. year: int64
- Levy: int64
- Mileage: float64
- Engine volume: float64
- Manufacturer: object
- Model: object
- Fuel type: object
- Leather interior: object
- Gear box type: object
- Category: object
- Price: int64

Valores nulos por columna:
- Prod. year: 0
- Levy: 0
- Mileage: 0
- Engine volume: 0
- Manufacturer: 0
- Model: 0
- Fuel type: 0
- Leather interior: 0
- Gear box type: 0
- Category: 0
- Price: 0

Estadísticas numéricas:
       Prod. year       Levy       Mileage  Engine volume         Price
count   15389.000  15389.000  1.538900e+04      15389.000  1.538900e+04
mean     2010.944   1126.826  1.578701e+06          2.308  1.894767e+04
std         5.637    727.227  4.962160e+07          0.880  2.128972e+05
min      1939.000      0.000  0.000000e+00          0.000  1.000000e+00
25%      2009.000    707.000  6.991800e+04          1.800  5.331000e+03
50%      2012.000    891.000  1.254450e+0

## 7. Imprimir el reporte directamente

`imprimir_reporte(df)` ejecuta la generación y muestra el resultado directamente. Es útil para consultar rápidamente el resumen durante una ejecución interactiva.

In [7]:
reportes.imprimir_reporte(df)

REPORTE DEL DATAFRAME
Filas: 15389
Columnas: 11
Duplicados: 2579

Columnas y tipos:
- Prod. year: int64
- Levy: int64
- Mileage: float64
- Engine volume: float64
- Manufacturer: object
- Model: object
- Fuel type: object
- Leather interior: object
- Gear box type: object
- Category: object
- Price: int64

Valores nulos por columna:
- Prod. year: 0
- Levy: 0
- Mileage: 0
- Engine volume: 0
- Manufacturer: 0
- Model: 0
- Fuel type: 0
- Leather interior: 0
- Gear box type: 0
- Category: 0
- Price: 0

Estadísticas numéricas:
       Prod. year       Levy       Mileage  Engine volume         Price
count   15389.000  15389.000  1.538900e+04      15389.000  1.538900e+04
mean     2010.944   1126.826  1.578701e+06          2.308  1.894767e+04
std         5.637    727.227  4.962160e+07          0.880  2.128972e+05
min      1939.000      0.000  0.000000e+00          0.000  1.000000e+00
25%      2009.000    707.000  6.991800e+04          1.800  5.331000e+03
50%      2012.000    891.000  1.254450e+0

## 8. Guardar el reporte en un archivo de texto

`guardar_reporte(df, ruta)` crea los directorios necesarios y escribe el reporte en UTF-8. El archivo quedará en `src/sincronicas/reportes/fast_food_restaurants.txt` cuando se ejecute desde `src/sincronicas`.

In [8]:
ruta = reportes.guardar_reporte(
    df,
    "reportes/car_dealership.txt",
)

print(f"Reporte guardado en: {ruta}")

Reporte guardado en: reportes/car_dealership.txt
